In [0]:
# Importar librerías
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_selection import (
    VarianceThreshold, SelectKBest, RFE, RFECV, SelectFromModel,
    chi2, f_classif, mutual_info_regression, mutual_info_classif
)
from sklearn.model_selection import cross_val_score, KFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import Lasso, LassoCV, ElasticNet
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline

# Configurar visualizaciones
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Librerías importadas correctamente")

In [0]:
# Cargar datasets de la panadería
ruta_datos = '/Workspace/Users/cortega@uda.edu.ar/Laboratorio/Datasets/'

df_ventas = pd.read_csv(ruta_datos + 'ventas.csv')
df_clientes = pd.read_csv(ruta_datos + 'clientes.csv')

print("✅ Datasets cargados:")
print(f"   Ventas: {len(df_ventas):,} registros")
print(f"   Clientes: {len(df_clientes):,} registros")

# Preparar datos
df_ventas['fecha'] = pd.to_datetime(df_ventas['fecha'])
df_ventas = df_ventas.sort_values('fecha').reset_index(drop=True)

# Features temporales
df_ventas['dia_semana'] = df_ventas['fecha'].dt.dayofweek
df_ventas['mes'] = df_ventas['fecha'].dt.month
df_ventas['es_fin_semana'] = df_ventas['dia_semana'].isin([5, 6]).astype(int)

# Merge con clientes
df = df_ventas.merge(df_clientes, on='cliente_id', how='left')
df['segmento_encoded'] = df['segmento'].astype('category').cat.codes

print("\n✅ Features temporales creadas")
print(f"\nColumnas disponibles: {list(df.columns)[:10]}...")  # Mostrar primeras 10

In [0]:
print("="*80)
print("COMPARACIÓN DE MÉTODOS DE SELECCIÓN DE FEATURES")
print("="*80)

# Preparar datos
df_ml = df[df['cliente_id'].notna()].copy()

features = ['sucursal_id', 'dia_semana', 'mes', 'es_fin_semana', 'segmento_encoded']
X = df_ml[features]
y = df_ml['total']

print(f"\n📊 Dataset: {len(X):,} registros, {len(features)} features")
print(f"\nProbando diferentes métodos de selección...\n")

# Modelo base
model_base = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)

# 1. Baseline: Todas las features
scores_all = cross_val_score(model_base, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

# 2. Mutual Information (top 3)
selector_mi = SelectKBest(score_func=mutual_info_regression, k=3)
pipeline_mi = Pipeline([('selector', selector_mi), ('model', model_base)])
scores_mi = cross_val_score(pipeline_mi, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

# 3. RF Importance (top 3)
model_for_importance = RandomForestRegressor(n_estimators=100, random_state=42)
model_for_importance.fit(X, y)
selector_rf = SelectFromModel(model_for_importance, threshold=-np.inf, max_features=3)
pipeline_rf = Pipeline([('selector', selector_rf), ('model', model_base)])
scores_rf = cross_val_score(pipeline_rf, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)

# Resultados
print("Resultados:")
print(f"\n1️⃣ Todas las features ({len(features)}):")
print(f"   MAE: ${-scores_all.mean():.2f} ± ${scores_all.std():.2f}")

print(f"\n2️⃣ Mutual Information (top 3):")
print(f"   MAE: ${-scores_mi.mean():.2f} ± ${scores_mi.std():.2f}")
selector_mi.fit(X, y)
mi_features = X.columns[selector_mi.get_support()].tolist()
print(f"   Features: {mi_features}")

print(f"\n3️⃣ RF Importance (top 3):")
print(f"   MAE: ${-scores_rf.mean():.2f} ± ${scores_rf.std():.2f}")
selector_rf.fit(X, y)
rf_features = X.columns[selector_rf.get_support()].tolist()
print(f"   Features: {rf_features}")

print(f"\n" + "="*80)
print("CONCLUSIÓN")
print("="*80)

mejora_mi = (-scores_all.mean()) - (-scores_mi.mean())
mejora_rf = (-scores_all.mean()) - (-scores_rf.mean())

print(f"\nReducir de {len(features)} a 3 features:")
print(f"  Mutual Info: Mejora de ${mejora_mi:.2f} ({mejora_mi/-scores_all.mean()*100:.1f}%)")
print(f"  RF Importance: Mejora de ${mejora_rf:.2f} ({mejora_rf/-scores_all.mean()*100:.1f}%)")
print(f"\n💡 Menos features puede dar MEJOR rendimiento (evita overfitting)")

## ✅ Conclusiones

### 🎯 Resumen del Módulo

**Lo que aprendimos**:

1. ✅ **Por qué seleccionar features** (maldición de dimensionalidad, overfitting)
2. ✅ **Métodos de Filtro** (varianza, correlación, MI, Chi², ANOVA F)
3. ✅ **Métodos Wrapper** (RFE, RFECV, forward/backward selection)
4. ✅ **Métodos Embedded** (Lasso, RF importance, XGBoost)
5. ✅ **Permutation Importance** (más confiable que RF importance)
6. ✅ **Selección con CV** (RFECV, nested CV, evitar leakage)
7. ✅ **Casos prácticos** con features H3 y temporales
8. ✅ **Comparación** de métodos y trade-offs

---

### 💡 Mensajes Clave

1. 🔑 **Menos features ≠ peor rendimiento** - Puede mejorar (evita overfitting)
2. ⚠️ **No hay "mejor" método** - Depende del problema y recursos
3. 🎯 **Filter → Embedded → Wrapper**: Workflow incremental
4. ⏱️ **Balance**: Velocidad (Filter) vs. Rendimiento (Wrapper)
5. 🔄 **Siempre usar Pipeline**: Evita data leakage en CV
6. 📊 **Validar con múltiples métodos**: No confiar en uno solo

---

### 📦 Guía Rápida de Selección
**¿Qué método usar?**

- 🚀 **Exploración rápida** → Mutual Information, Correlación
- ⚖️ **Balance velocidad/rendimiento** → RF Importance, Lasso
- 🎯 **Máximo rendimiento** → RFECV, Permutation Importance
- 📊 **Muchas features (>10K)** → Filter methods
- 🔬 **Pocas features (<1K)** → Wrapper methods
- 🧠 **Interpretabilidad** → Lasso, Correlación

---

### 📚 Recursos Adicionales

- [Scikit-learn Feature Selection](https://scikit-learn.org/stable/modules/feature_selection.html)
- [Feature Selection for Machine Learning](https://machinelearningmastery.com/feature-selection-machine-learning-python/)
- [Permutation Importance](https://christophm.github.io/interpretable-ml-book/feature-importance.html)

---

## 🎓 ¡Felicitaciones!

**Has completado el módulo de Selección de Características.**

Ahora puedes:
- ✅ Identificar cuándo necesitas selección de features
- ✅ Aplicar 10+ métodos diferentes (filtro, wrapper, embedded)
- ✅ Usar RFECV para encontrar N óptimo automáticamente
- ✅ Calcular Permutation Importance confiable
- ✅ Integrar selección con validación cruzada (Pipeline)
- ✅ Seleccionar features H3 y temporales correctamente
- ✅ Comparar métodos y elegir el mejor para tu caso

**Próximo paso**: Notebook Práctico con ejercicios hands-on.

---

**Universidad del Aconcagua**  
**Laboratorio (Herramientas)**  
**Mendoza, Argentina**

## 7️⃣ Casos Prácticos: Features H3 y Temporales

### 🗺️ Caso 1: Selección de Features H3

**Problema**: Dataset con **múltiples resoluciones H3** (res 5, 7, 9) - **features redundantes**.

**Estrategia**:
1. **Calcular correlación** entre resoluciones
2. **Seleccionar la resolución óptima** con CV
3. **Eliminar features H3 redundantes**

**Ejemplo**:
```python
import h3
import pandas as pd
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

# Dataset con múltiples resoluciones H3
df['h3_res5'] = df.apply(lambda row: h3.geo_to_h3(row['lat'], row['lon'], 5), axis=1)
df['h3_res7'] = df.apply(lambda row: h3.geo_to_h3(row['lat'], row['lon'], 7), axis=1)
df['h3_res9'] = df.apply(lambda row: h3.geo_to_h3(row['lat'], row['lon'], 9), axis=1)

# Encodear H3 (label encoding)
for col in ['h3_res5', 'h3_res7', 'h3_res9']:
    df[col + '_encoded'] = df[col].astype('category').cat.codes

# Comparar importancia por resolución
h3_features = ['h3_res5_encoded', 'h3_res7_encoded', 'h3_res9_encoded']
X_h3 = df[h3_features]
y = df['ventas']

# Mutual information
mi_scores = mutual_info_regression(X_h3, y)
for feature, score in zip(h3_features, mi_scores):
    print(f"{feature}: {score:.4f}")

# Probar cada resolución con CV
model = RandomForestRegressor(n_estimators=100, random_state=42)

for feature in h3_features:
    X_single = df[[feature]]
    scores = cross_val_score(model, X_single, y, cv=5, scoring='neg_mean_absolute_error')
    print(f"{feature}: MAE = ${-scores.mean():.2f}")

# Resultado: Seleccionar la resolución con mejor balance (usualmente res 7)
```

**Recomendación**: Para ventas urbanas, **resolución 7** (~5 km²) suele ser óptima.

---

### 📅 Caso 2: Selección de Features Temporales

**Problema**: Dataset con **muchas features temporales** (día, mes, semana, trimestre, etc.) - **redundantes**.

**Estrategia**:
1. **Probar combinaciones** de features temporales
2. **Eliminar features temporales correlacionadas**
3. **Mantener solo las más informativas**

**Ejemplo**:
```python
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# Features temporales
df['dia_semana'] = df['fecha'].dt.dayofweek
df['mes'] = df['fecha'].dt.month
df['dia_mes'] = df['fecha'].dt.day
df['trimestre'] = df['fecha'].dt.quarter
df['semana_anio'] = df['fecha'].dt.isocalendar().week
df['es_fin_semana'] = df['dia_semana'].isin([5, 6]).astype(int)

temporal_features = ['dia_semana', 'mes', 'dia_mes', 'trimestre', 'semana_anio', 'es_fin_semana']

# Seleccionar top temporal features
X_temporal = df[temporal_features]
y = df['ventas']

selector = SelectKBest(score_func=mutual_info_regression, k=3)
X_selected = selector.fit_transform(X_temporal, y)

selected_features = [temporal_features[i] for i in range(len(temporal_features)) if selector.get_support()[i]]
print(f"Features temporales seleccionadas: {selected_features}")

# Típicamente: ['dia_semana', 'mes', 'es_fin_semana'] son las más importantes
```

---

### 🔄 Caso 3: Features H3 + Temporales + Otras

**Dataset completo de panadería**:
```python
# Todas las features
features = [
    # Temporales
    'dia_semana', 'mes', 'es_fin_semana',
    # Geoespaciales
    'h3_res7_encoded', 'sucursal_id',
    # Cliente
    'segmento_encoded', 'cliente_es_frecuente',
    # Producto
    'categoria_producto', 'precio_promedio'
]

X = df[features]
y = df['total']

# Pipeline completo
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('selector', SelectKBest(score_func=mutual_info_regression, k=5)),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42))
])

# CV
from sklearn.model_selection import cross_val_score
scores = cross_val_score(pipeline, X, y, cv=5, scoring='neg_mean_absolute_error')

print(f"MAE: ${-scores.mean():.2f} ± ${scores.std():.2f}")
```

---

### 💡 Recomendaciones para Features H3

1. ✅ **No uses múltiples resoluciones** a la vez (son redundantes)
2. ✅ **Resolución 7** (∼5 km²) es buena para la mayoría de casos urbanos
3. ✅ **Resolución 5** (∼250 km²) para análisis regional
4. ✅ **Resolución 9** (∼0.1 km²) para análisis hiper-local
5. ✅ **Combina H3 con otras features geoespaciales** (ciudad, zona, etc.)

---

## 8️⃣ Comparación Completa de Métodos

### 📊 Tabla Comparativa General

| Método | Tipo | Velocidad | Rendimiento | Interpretabilidad | Escalabilidad | Cuándo Usar |
|---------|------|-----------|-------------|-------------------|---------------|---------------|
| **Varianza** | Filter | ⭐⭐⭐⭐⭐ | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Features constantes |
| **Correlación** | Filter | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Regresión lineal |
| **Chi²** | Filter | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Clasificación, features categóricas |
| **Mutual Info** | Filter | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | Relaciones no lineales |
| **ANOVA F** | Filter | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Clasificación, features continuas |
| **RFE** | Wrapper | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐ | Máximo rendimiento, pocos features |
| **RFECV** | Wrapper | ⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐ | Encuentra N óptimo automáticamente |
| **Lasso** | Embedded | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | Regresión lineal, interpretabilidad |
| **RF Importance** | Embedded | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐⭐ | ⭐⭐⭐ | Relaciones no lineales, rápido |
| **Permutation** | Post-hoc | ⭐⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ | ⭐⭐ | Máxima confiabilidad |

---

### ⚖️ Trade-offs Principales

**Velocidad vs. Rendimiento**:
```
Filter (rápido, rendimiento bajo)
  ↓
Embedded (balance)
  ↓
Wrapper (lento, rendimiento alto)
```

**Escalabilidad vs. Precisión**:
```
Filter: Escala a millones de features
Embedded: Escala a decenas de miles
Wrapper: Escala a cientos/miles
```

---

### 🧩 Árbol de Decisión: ¿Qué Método Usar?

```
¿Cuántas features tienes?
│
├── > 10,000 → Filter Methods (Mutual Info, Chi²)
│
├── 1,000 - 10,000
│   └── ¿Tiempo es crítico?
│       ├── Sí → Embedded (RF Importance, Lasso)
│       └── No → RFECV
│
└── < 1,000
    └── ¿Quieres máximo rendimiento?
        ├── Sí → RFECV (wrapper)
        └── No → Embedded (RF Importance, Lasso)
```

---

### 📊 Experimento Comparativo

**Dataset**: Ventas de panadería (50,000 registros, 20 features)

| Método | Features Seleccionadas | Tiempo | MAE | Comentario |
|---------|------------------------|--------|-----|------------|
| **Todas las features** | 20 | - | $15.20 | Baseline |
| **Mutual Info (top 10)** | 10 | 2s | $14.80 | Rápido, bueno |
| **Lasso** | 8 | 5s | $14.50 | Interpretable |
| **RF Importance (top 10)** | 10 | 10s | $14.30 | Buen balance |
| **RFECV** | 12 | 60s | **$14.10** | Mejor rendimiento |
| **Permutation (top 10)** | 10 | 45s | $14.15 | Más confiable |

🎯 **Recomendación**: Usar **RF Importance** para exploración rápida, **RFECV** para modelo final.

---

### 💡 Workflow Recomendado

**Fase 1: Exploración Rápida** (⏱️ Minutos)
1. Eliminar features con **baja varianza**
2. Calcular **correlaciones** (eliminar redundantes)
3. Calcular **Mutual Information** (top 20-30)

**Fase 2: Selección Iterativa** (⏱️ Horas)
4. Entrenar **Random Forest** con features de Fase 1
5. Usar **RF Importance** o **Lasso** (top 10-15)
6. Evaluar con **validación cruzada**

**Fase 3: Optimización Final** (⏱️ Días)
7. **RFECV** para encontrar N óptimo
8. **Permutation Importance** para validar
9. **Nested CV** para evaluación final

---

## 9️⃣ Mejores Prácticas de Selección de Features

### ✅ DO: Buenas Prácticas

#### 1. **Siempre Escalar Datos Antes de Selección**
```python
# ✅ BIEN: Escalar antes de Lasso o VarianceThreshold
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = Lasso(alpha=1.0)
model.fit(X_scaled, y)

# ❌ MAL: Sin escalar
model.fit(X, y)  # Features con escalas diferentes son penalizadas injustamente
```

#### 2. **Usar Pipeline para Evitar Data Leakage**
```python
# ✅ BIEN: Selección dentro del CV
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, mutual_info_regression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=mutual_info_regression, k=10)),
    ('model', RandomForestRegressor())
])

scores = cross_val_score(pipeline, X, y, cv=5)

# ❌ MAL: Selección antes del CV
selector = SelectKBest(k=10)
X_selected = selector.fit_transform(X, y)  # Data leakage!
scores = cross_val_score(model, X_selected, y, cv=5)
```

#### 3. **Probar Múltiples Métodos**
```python
# ✅ BIEN: Comparar múltiples métodos
methods = {
    'mutual_info': SelectKBest(score_func=mutual_info_regression, k=10),
    'lasso': SelectFromModel(Lasso(alpha=1.0)),
    'rf_importance': SelectFromModel(RandomForestRegressor(n_estimators=100))
}

for name, selector in methods.items():
    pipeline = Pipeline([('selector', selector), ('model', RandomForestRegressor())])
    scores = cross_val_score(pipeline, X, y, cv=5, scoring='r2')
    print(f"{name}: R² = {scores.mean():.4f}")
```

#### 4. **Documentar Features Seleccionadas**
```python
# ✅ BIEN: Guardar y documentar
selected_features = X.columns[selector.get_support()].tolist()

import json
with open('selected_features.json', 'w') as f:
    json.dump({
        'features': selected_features,
        'method': 'RFECV',
        'score': best_score,
        'date': '2026-07-27'
    }, f, indent=2)
```

#### 5. **Validar con Nested CV**
```python
# ✅ BIEN: Nested CV para evaluación realista
# (ver sección 6 para implementación completa)
```

---

### ❌ DON'T: Errores Comunes

#### 1. **NO Seleccionar en Todo el Dataset Antes de CV**
```python
# ❌ MAL
X_selected = selector.fit_transform(X, y)  # Usa info de test
scores = cross_val_score(model, X_selected, y, cv=5)  # Leakage!
```

#### 2. **NO Usar Solo RF Importance con Features Correlacionadas**
```python
# ❌ MAL: RF importance sesgado con features correlacionadas
importances = model.feature_importances_

# ✅ BIEN: Usar Permutation Importance
from sklearn.inspection import permutation_importance
result = permutation_importance(model, X_test, y_test, n_repeats=10)
```

#### 3. **NO Eliminar Features Sin Analizar**
```python
# ❌ MAL: Eliminar sin revisar
X_reduced = X.drop(['feature_1', 'feature_2'], axis=1)

# ✅ BIEN: Analizar correlaciones y importancias primero
correlations = df.corr()['target'].abs().sort_values(ascending=False)
print(correlations[['feature_1', 'feature_2']])
```

#### 4. **NO Usar Lasso Sin Escalar**
```python
# ❌ MAL
model = Lasso(alpha=1.0)
model.fit(X, y)  # Features con mayor escala son sobre-penalizadas

# ✅ BIEN
X_scaled = StandardScaler().fit_transform(X)
model.fit(X_scaled, y)
```

#### 5. **NO Confiar en un Solo Método**
```python
# ❌ MAL: Solo usar un método
selector = SelectKBest(k=10)

# ✅ BIEN: Validar con múltiples métodos
mi_features = SelectKBest(mutual_info_regression, k=10).fit(X, y).get_support()
rf_features = SelectFromModel(RandomForestRegressor()).fit(X, y).get_support()

# Features seleccionadas por ambos métodos
common_features = X.columns[mi_features & rf_features].tolist()
```

---

### 📊 Cuántas Features Seleccionar?

**Reglas prácticas**:

| Tamaño Dataset | Features Iniciales | Features Objetivo |
|-----------------|--------------------| ------------------|
| < 1,000 | 100 | 5-10 |
| 1,000 - 10,000 | 100-1,000 | 10-30 |
| 10,000 - 100,000 | 1,000-10,000 | 30-100 |
| > 100,000 | > 10,000 | 100-500 |

⚠️ **Mejor enfoque**: Usar **RFECV** o **validación cruzada** para encontrar el número óptimo.

---

### 💡 Tips Finales

1. ✅ **Empieza con filtros** (rápido) para reducción inicial
2. ✅ **Usa embedded** (RF, Lasso) para selección intermedia
3. ✅ **Finaliza con wrapper** (RFECV) si tiempo lo permite
4. ✅ **Valida con Permutation Importance** antes de producción
5. ✅ **Menos es más**: Prefiere **modelos simples e interpretables**
6. ✅ **Documenta tu proceso**: Qué features, por qué, cuándo
7. ✅ **Monitorea en producción**: Features pueden perder importancia con el tiempo

---

## 5️⃣ Permutation Importance

### 📖 Concepto

**Permutation Importance** mide **cuánto empeora** el rendimiento del modelo cuando se **permutan (mezclan) los valores** de una feature.

**Idea**:
1. Entrenar modelo
2. Calcular rendimiento baseline
3. Para cada feature:
   - **Permutar** sus valores aleatoriamente
   - Calcular nuevo rendimiento
   - Importancia = baseline - nuevo rendimiento
4. Feature con mayor caída = más importante

**Ventaja**:
- ✅ **Model-agnostic**: Funciona con **cualquier modelo**
- ✅ **Más confiable** que feature importance de árboles
- ✅ **Considera interacciones** entre features

---

### 💻 Implementación

```python
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Split datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Calcular permutation importance
result = permutation_importance(
    model, X_test, y_test, 
    n_repeats=10,  # Número de permutaciones
    random_state=42,
    n_jobs=-1
)

# Resultados
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': result.importances_mean,
    'std': result.importances_std
}).sort_values('importance', ascending=False)

print(importances)
```

---

### 📊 Visualización

```python
import matplotlib.pyplot as plt

# Barplot con error bars
plt.figure(figsize=(10, 6))
plt.barh(
    importances['feature'][:15], 
    importances['importance'][:15],
    xerr=importances['std'][:15]  # Error bars
)
plt.xlabel('Permutation Importance')
plt.title('Top 15 Features por Permutation Importance')
plt.gca().invert_yaxis()
plt.show()
```

---

### ⚖️ RF Feature Importance vs. Permutation Importance

**Diferencias**:

| Aspecto | RF Importance | Permutation Importance |
|---------|---------------|-------------------------|
| **Cálculo** | Durante entrenamiento | Después de entrenar |
| **Velocidad** | ⭐⭐⭐⭐⭐ Rápido | ⭐⭐⭐ Más lento |
| **Confiabilidad** | Sesgado con features correlacionadas | ✅ Más confiable |
| **Aplicabilidad** | Solo modelos tree-based | ✅ Cualquier modelo |

**Problema de RF Importance**: Sobreestima features con **alta cardinalidad** (muchos valores únicos).

**Ejemplo**:
```python
# Feature importance (sesgado)
rf_importance = model.feature_importances_

# Permutation importance (más confiable)
perm_importance = result.importances_mean

# Comparar
comparison = pd.DataFrame({
    'feature': X.columns,
    'rf_importance': rf_importance,
    'perm_importance': perm_importance
}).sort_values('perm_importance', ascending=False)

print(comparison)
```

---

### 🎯 Cuándo Usar Permutation Importance

✅ **Usar cuando**:
- Quieres **importancia más confiable**
- Tienes features **correlacionadas**
- Modelo es **black-box** (KNN, SVM, redes neuronales)
- Tiempo no es crítico (más lento que RF importance)

❌ **No usar cuando**:
- Dataset es **muy grande** (lento)
- Necesitas **velocidad**

---

## 6️⃣ Selección de Features con Validación Cruzada

### 📖 Concepto

**Problema**: ¿Cómo saber cuántas features seleccionar?

**Solución**: Usar **validación cruzada** para encontrar el número óptimo.

---

### 📊 Método 1: RFECV (RFE + CV)

**Ya visto antes**, pero repasemos:

```python
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

# RFECV con K-Fold CV
model = RandomForestRegressor(n_estimators=100, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

selector = RFECV(
    estimator=model,
    step=1,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

selector.fit(X, y)

print(f"Número óptimo de features: {selector.n_features_}")
print(f"Features: {X.columns[selector.support_].tolist()}")

# Visualizar curva
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(selector.cv_results_['mean_test_score']) + 1),
         -selector.cv_results_['mean_test_score'])  # Negativo porque es neg_mae
plt.xlabel('Número de Features')
plt.ylabel('MAE (CV)')
plt.title('RFECV: MAE vs. Número de Features')
plt.axvline(x=selector.n_features_, color='red', linestyle='--', label=f'Optimal = {selector.n_features_}')
plt.legend()
plt.grid(True)
plt.show()
```

---

### 📊 Método 2: SelectFromModel + CV

**Idea**: Probar diferentes thresholds con CV.

```python
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np

# Entrenar modelo para obtener importances
model_full = RandomForestRegressor(n_estimators=100, random_state=42)
model_full.fit(X, y)

# Probar diferentes thresholds
thresholds = np.linspace(0, model_full.feature_importances_.max(), 20)
results = []

for threshold in thresholds:
    # Seleccionar features
    selector = SelectFromModel(model_full, threshold=threshold, prefit=True)
    X_selected = selector.transform(X)
    
    # Skip si no hay features
    if X_selected.shape[1] == 0:
        continue
    
    # Evaluar con CV
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    scores = cross_val_score(model, X_selected, y, cv=5, scoring='neg_mean_absolute_error')
    
    results.append({
        'threshold': threshold,
        'n_features': X_selected.shape[1],
        'mae': -scores.mean()
    })

results_df = pd.DataFrame(results)

# Encontrar óptimo
best_idx = results_df['mae'].idxmin()
best_result = results_df.loc[best_idx]

print(f"Threshold óptimo: {best_result['threshold']:.4f}")
print(f"Número óptimo de features: {int(best_result['n_features'])}")
print(f"MAE: ${best_result['mae']:.2f}")

# Visualizar
plt.figure(figsize=(10, 6))
plt.plot(results_df['n_features'], results_df['mae'], marker='o')
plt.xlabel('Número de Features')
plt.ylabel('MAE (CV)')
plt.title('MAE vs. Número de Features')
plt.axvline(x=best_result['n_features'], color='red', linestyle='--', label='Optimal')
plt.legend()
plt.grid(True)
plt.show()
```

---

### 📊 Método 3: Nested CV para Selección + Evaluación

**Problema**: Usar el mismo CV para selección y evaluación → **overfitting**.

**Solución**: **Nested CV** (CV anidado).

**Estructura**:
- **Outer CV**: Evaluación del rendimiento
- **Inner CV**: Selección de features

```python
from sklearn.model_selection import cross_val_score, KFold
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestRegressor

# Outer CV (evaluación)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Inner CV (selección)
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)

scores = []

for train_idx, test_idx in outer_cv.split(X):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Selección en train (inner CV)
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    selector = RFECV(estimator=model, cv=inner_cv, scoring='r2', n_jobs=-1)
    selector.fit(X_train, y_train)
    
    X_train_selected = selector.transform(X_train)
    X_test_selected = selector.transform(X_test)
    
    # Entrenar modelo final con features seleccionadas
    model_final = RandomForestRegressor(n_estimators=100, random_state=42)
    model_final.fit(X_train_selected, y_train)
    
    # Evaluar en test
    score = model_final.score(X_test_selected, y_test)
    scores.append(score)
    
    print(f"Fold: R² = {score:.4f}, Features = {selector.n_features_}")

print(f"\nR² promedio: {np.mean(scores):.4f} ± {np.std(scores):.4f}")
```

💡 **Nested CV** da una estimación **más realista** del rendimiento.

---

### ⚠️ Errores Comunes

❌ **Error 1**: Seleccionar features en todo el dataset antes de CV
```python
# MAL
selector = SelectKBest(k=10)
X_selected = selector.fit_transform(X, y)  # Usa TODO el dataset
scores = cross_val_score(model, X_selected, y, cv=5)  # Data leakage!
```

✅ **Correcto**: Seleccionar dentro del CV
```python
# BIEN
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('selector', SelectKBest(k=10)),
    ('model', RandomForestRegressor())
])

scores = cross_val_score(pipeline, X, y, cv=5)  # Selección en cada fold
```

---

## 3️⃣ Métodos Wrapper (Wrapper Methods)

### 📖 Concepto

**Métodos wrapper** evalúan **subconjuntos de features** entrenando un modelo y midiendo su rendimiento.

**Proceso**:
1. Entrenar modelo con subconjunto de features
2. Medir rendimiento (accuracy, MAE, etc.)
3. Probar otro subconjunto
4. Repetir hasta encontrar el mejor

**Ventajas**:
- ✅ **Considera interacciones**: Entre features
- ✅ **Optimiza para el modelo específico**
- ✅ **Mejor rendimiento**: Que métodos de filtro

**Desventajas**:
- ❌ **Muy lento**: Entrenan muchos modelos
- ❌ **Riesgo de overfitting**: Sobre el conjunto de validación
- ❌ **No escalable**: Impracticable con miles de features

---

### 🔼 Método 1: Forward Selection (Selección Hacia Adelante)

**Proceso**:
1. Empezar sin features
2. Probar **agregar cada feature** restante
3. Seleccionar la que **mejora más** el rendimiento
4. Repetir hasta que no haya mejora

**Visualización**:
```
Iteración 1: [] → [F1] (mejor)
Iteración 2: [F1] → [F1, F3] (mejor)
Iteración 3: [F1, F3] → [F1, F3, F7] (mejor)
Iteración 4: [F1, F3, F7] → No mejora → PARAR
```

**Implementación (manual)**:
```python
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestRegressor

def forward_selection(X, y, max_features=10):
    selected_features = []
    remaining_features = list(X.columns)
    
    model = RandomForestRegressor(n_estimators=50, random_state=42)
    best_score = -float('inf')
    
    for i in range(max_features):
        scores = []
        
        # Probar cada feature restante
        for feature in remaining_features:
            features_to_test = selected_features + [feature]
            X_subset = X[features_to_test]
            
            # Evaluar con CV
            score = cross_val_score(model, X_subset, y, cv=5, scoring='r2').mean()
            scores.append((feature, score))
        
        # Seleccionar mejor feature
        best_feature, current_best_score = max(scores, key=lambda x: x[1])
        
        # Si no mejora, parar
        if current_best_score <= best_score:
            break
        
        # Agregar feature
        selected_features.append(best_feature)
        remaining_features.remove(best_feature)
        best_score = current_best_score
        
        print(f"Iteración {i+1}: Agregada {best_feature}, R² = {best_score:.4f}")
    
    return selected_features

# Usar
selected = forward_selection(X, y, max_features=5)
print(f"\nFeatures seleccionadas: {selected}")
```

---

### 🔽 Método 2: Backward Elimination (Eliminación Hacia Atrás)

**Proceso**:
1. Empezar con **todas las features**
2. Probar **eliminar cada feature**
3. Eliminar la que **afecta menos** al rendimiento
4. Repetir hasta que eliminar empeore mucho

**Visualización**:
```
Iteración 1: [F1, F2, F3, F4, F5] → [F1, F2, F3, F5] (eliminar F4)
Iteración 2: [F1, F2, F3, F5] → [F1, F3, F5] (eliminar F2)
Iteración 3: [F1, F3, F5] → Eliminar empeora mucho → PARAR
```

⚠️ **Problema**: Más lento que forward con muchas features.

---

### 🔄 Método 3: Recursive Feature Elimination (RFE)

**Proceso**:
1. Entrenar modelo con todas las features
2. Calcular **importancia** de cada feature
3. **Eliminar la menos importante**
4. Repetir hasta tener N features deseadas

**Ventaja**: Más eficiente que backward elimination.

**Implementación**:
```python
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

# RFE para seleccionar top 10 features
model = RandomForestRegressor(n_estimators=100, random_state=42)
selector = RFE(estimator=model, n_features_to_select=10, step=1)

X_selected = selector.fit_transform(X, y)

# Ver features seleccionadas
selected_features = X.columns[selector.support_].tolist()
print(f"Features seleccionadas: {selected_features}")

# Ver ranking de features
ranking = pd.DataFrame({
    'feature': X.columns,
    'ranking': selector.ranking_
}).sort_values('ranking')

print("\nRanking de features:")
print(ranking)
```

**RFE con Validación Cruzada (RFECV)**:
```python
from sklearn.feature_selection import RFECV

# RFECV encuentra automáticamente el número óptimo
selector = RFECV(
    estimator=model, 
    step=1, 
    cv=5,  # 5-fold CV
    scoring='r2',
    n_jobs=-1
)

selector.fit(X, y)

print(f"Número óptimo de features: {selector.n_features_}")
print(f"Features seleccionadas: {X.columns[selector.support_].tolist()}")

# Visualizar
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(selector.cv_results_['mean_test_score']) + 1), 
         selector.cv_results_['mean_test_score'])
plt.xlabel('Número de Features')
plt.ylabel('R² Score (CV)')
plt.title('RFECV: Rendimiento vs. Número de Features')
plt.grid(True)
plt.show()
```

---

### ⚖️ Comparación de Métodos Wrapper

| Método | Complejidad | Resultado | Cuándo Usar |
|---------|-------------|-----------|---------------|
| **Forward Selection** | O(N²) | Bueno | Pocas features esperadas |
| **Backward Elimination** | O(N²) | Bueno | Muchas features importantes |
| **RFE** | O(N) | Muy bueno | **Recomendado** (más eficiente) |
| **RFECV** | O(N × K) | Excelente | Mejor opción (encuentra N óptimo) |

N = número de features, K = número de folds en CV

---

### 🎯 Cuándo Usar Métodos Wrapper

✅ **Usar cuando**:
- Dataset **mediano** (< 1000 features)
- Tiempo de entrenamiento **no es crítico**
- Quieres **máximo rendimiento**
- Tienes recursos computacionales

❌ **No usar cuando**:
- Dataset **muy grande** (> 10,000 features)
- Tiempo es **crítico**
- Recursos computacionales **limitados**

---

## 4️⃣ Métodos Embedded (Embedded Methods)

### 📖 Concepto

**Métodos embedded** realizan selección de features **durante el entrenamiento del modelo**.

**Ventaja clave**: Balance entre **velocidad** (más rápido que wrapper) y **rendimiento** (mejor que filtro).

---

### 🎯 Método 1: Regularización Lasso (L1)

**Idea**: Lasso **penaliza coeficientes** y los **fuerza a cero**.

**Fórmula**:
```
Loss = MSE + α × ∑|wᵢ|
```

- α alto → más coeficientes = 0 → menos features
- α bajo → menos penalización → más features

**Implementación**:
```python
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler

# Escalar datos (importante para Lasso)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Lasso con penalización fuerte
model = Lasso(alpha=1.0, random_state=42)
model.fit(X_scaled, y)

# Features con coeficiente != 0
selected_features = X.columns[model.coef_ != 0].tolist()
print(f"Features seleccionadas: {selected_features}")
print(f"Número: {len(selected_features)}")

# Visualizar coeficientes
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(X.columns, model.coef_)
plt.xlabel('Coeficiente')
plt.title('Coeficientes Lasso')
plt.axvline(x=0, color='red', linestyle='--')
plt.show()
```

**LassoCV** (encuentra α óptimo con CV):
```python
from sklearn.linear_model import LassoCV

# LassoCV prueba múltiples alphas
model = LassoCV(cv=5, random_state=42, n_jobs=-1)
model.fit(X_scaled, y)

print(f"Alpha óptimo: {model.alpha_}")
print(f"R² Score: {model.score(X_scaled, y):.4f}")

selected_features = X.columns[model.coef_ != 0].tolist()
print(f"Features seleccionadas: {len(selected_features)}")
```

---

### 🎯 Método 2: Ridge (L2) vs. ElasticNet

**Ridge (L2)**:
- **No elimina features** (coeficientes pequeños pero != 0)
- Útil para **regularización**, no para selección

**ElasticNet** (L1 + L2):
- Combina Lasso y Ridge
- Balance entre **selección** y **estabilidad**

```python
from sklearn.linear_model import ElasticNet

model = ElasticNet(alpha=1.0, l1_ratio=0.5, random_state=42)
model.fit(X_scaled, y)

selected_features = X.columns[model.coef_ != 0].tolist()
print(f"Features seleccionadas: {selected_features}")
```

**l1_ratio**:
- `l1_ratio=1`: Solo L1 (Lasso)
- `l1_ratio=0`: Solo L2 (Ridge)
- `l1_ratio=0.5`: 50% L1 + 50% L2

---

### 🎯 Método 3: Tree-Based Feature Importance

**Idea**: Árboles de decisión calculan **importancia** de features automáticamente.

**Métrica**: Reducción promedio de impureza (Gini o entropía).

#### Random Forest Feature Importance

```python
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Feature importance
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances)

# Visualizar
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(importances['feature'][:15], importances['importance'][:15])
plt.xlabel('Importancia')
plt.title('Top 15 Features por Importancia (Random Forest)')
plt.gca().invert_yaxis()
plt.show()

# Seleccionar top 10
top_features = importances['feature'][:10].tolist()
X_selected = X[top_features]
```

#### XGBoost Feature Importance

```python
import xgboost as xgb

model = xgb.XGBRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Feature importance
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importances.head(15))

# XGBoost tiene plot_importance built-in
xgb.plot_importance(model, max_num_features=15, importance_type='weight')
plt.show()
```

**Tipos de importancia en XGBoost**:
- `weight`: Número de veces que se usa la feature
- `gain`: Ganancia promedio al usar la feature
- `cover`: Número promedio de muestras afectadas

---

### 🎯 Método 4: SelectFromModel

**Idea**: Seleccionar features usando **cualquier modelo con feature importance**.

```python
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor

# Entrenar modelo
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

# Seleccionar features con importancia > threshold
selector = SelectFromModel(model, threshold='median')  # median, mean, o valor específico
selector.fit(X, y)

X_selected = selector.transform(X)

# Ver features seleccionadas
selected_features = X.columns[selector.get_support()].tolist()
print(f"Features seleccionadas: {selected_features}")
print(f"Número: {len(selected_features)}")
```

**Thresholds comunes**:
- `'median'`: Selecciona 50% de features
- `'mean'`: Selecciona features sobre la media
- `'1.5*mean'`: Selecciona solo las muy importantes
- `0.01`: Importancia > 0.01

---

### ⚖️ Comparación de Métodos Embedded

| Método | Tipo Modelo | Velocidad | Interpretabilidad | Cuándo Usar |
|---------|-------------|-----------|-------------------|---------------|
| **Lasso** | Lineal | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐⭐ | Relaciones lineales |
| **ElasticNet** | Lineal | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ | Features correlacionadas |
| **RF Importance** | Tree-based | ⭐⭐⭐⭐ | ⭐⭐⭐ | Relaciones no lineales |
| **XGBoost Importance** | Tree-based | ⭐⭐⭐ | ⭐⭐⭐ | Máximo rendimiento |

---

### 🎯 Cuándo Usar Métodos Embedded

✅ **Usar cuando**:
- Quieres **balance** entre velocidad y rendimiento
- Tienes modelo con **feature importance** (RF, XGBoost)
- Dataset **mediano a grande** (1000-10000 features)
- Buscas **interpretabilidad** (Lasso)

❌ **No usar cuando**:
- Modelo no tiene feature importance (KNN, SVM sin kernel)
- Quieres máximo rendimiento (usar wrapper)

---

# 🎯 Selección de Características (Feature Selection)
## Material Complementario - Laboratorio (Herramientas)
### Universidad del Aconcagua - Mendoza, Argentina

---

### 🎯 Objetivos de Aprendizaje

1. Comprender **por qué seleccionar features** (maldición de la dimensionalidad)
2. Dominar **métodos de filtro** (varianza, correlación, MI)
3. Aplicar **métodos wrapper** (RFE, forward/backward selection)
4. Usar **métodos embedded** (Lasso, tree-based importance)
5. Analizar **importancia de features** (permutation, SHAP)
6. Integrar selección con **validación cruzada**
7. Aplicar técnicas a **features H3 y temporales**

### 📁 Contenido

1. ¿Por qué Selección de Características?
2. Métodos de Filtro (Filter Methods)
3. Métodos Wrapper (Wrapper Methods)
4. Métodos Embedded (Embedded Methods)
5. Feature Importance y Permutation Importance
6. Selección con Validación Cruzada
7. Casos Prácticos: Features H3 y Temporales
8. Comparación de Métodos
9. Mejores Prácticas

### ⏱️ Duración Estimada: 2-3 horas

---

## 1️⃣ ¿Por qué Necesitamos Selección de Características?

### 🚨 Problemas con Muchas Features

**Escenario típico**:
```python
# Dataset con 100 features
X_train.shape  # (10000, 100)
```

❌ **Problemas**:

#### 1. **Maldición de la Dimensionalidad** (Curse of Dimensionality)

- Más features → espacio de búsqueda **exponencialmente mayor**
- Datos se vuelven **dispersos** (sparse)
- Se necesita **mucho más datos** para entrenar bien

**Ejemplo**:
```
10 features → necesitas ~1,000 registros
100 features → necesitas ~100,000 registros
1,000 features → necesitas ~10,000,000 registros
```

#### 2. **Overfitting**

- Modelo aprende **ruido** en lugar de patrones reales
- Buen rendimiento en train, **mal rendimiento en test**

#### 3. **Tiempo de Entrenamiento**

- Más features → más tiempo de cómputo
- 100 features puede ser **10x más lento** que 10 features

#### 4. **Interpretabilidad**

- Difícil explicar un modelo con 100 features
- Stakeholders quieren modelos **simples y entendibles**

#### 5. **Features Redundantes o Irrelevantes**

- **Redundantes**: Correlacionadas entre sí (ej: peso en kg y peso en lb)
- **Irrelevantes**: No aportan información (ej: ID de cliente para predecir ventas)

---

### ✅ Beneficios de Selección de Features

1. ✅ **Mejor rendimiento**: Menos overfitting
2. ✅ **Más rápido**: Menor tiempo de entrenamiento e inferencia
3. ✅ **Más interpretable**: Modelos más simples
4. ✅ **Menos datos necesarios**: Evita maldición de dimensionalidad
5. ✅ **Menos almacenamiento**: Menos features en producción

---

### 📊 Ejemplo Visual

**Dataset original**: 50 features
```
[edad, ingreso, ciudad, producto_1, producto_2, ..., producto_45, ruido_1, ruido_2, ruido_3]
```

**Después de selección**: 10 features importantes
```
[edad, ingreso, ciudad, producto_3, producto_7, producto_12, producto_20, producto_31, producto_40, producto_45]
```

**Resultados**:
- Accuracy: 85% → **87%** (✅ mejor)
- Tiempo de entrenamiento: 5 min → **30 seg** (✅ 5x más rápido)
- Interpretabilidad: ❌ Compleja → ✅ **Simple**

---

### 📦 Tipos de Features

| Tipo | Descripción | Acción |
|------|-------------|----------|
| **⭐ Relevantes** | Correlacionadas con el target | ✅ **Mantener** |
| **🔄 Redundantes** | Correlacionadas entre sí | ❌ **Eliminar una** |
| **🚫 Irrelevantes** | No aportan información | ❌ **Eliminar** |
| **🎲 Ruido** | Patrones aleatorios sin sentido | ❌ **Eliminar** |

---

### 🧠 ¿Selección vs. Extracción de Features?

**Feature Selection** (lo que veremos):
- **Selecciona subconjunto** de features existentes
- Mantiene features **originales e interpretables**
- Métodos: filtro, wrapper, embedded

**Feature Extraction** (PCA, t-SNE, etc.):
- **Crea nuevas features** combinando las originales
- Features transformadas (**menos interpretables**)
- Métodos: PCA, LDA, autoencoders

🎯 **Este módulo se enfoca en Feature Selection.**

---

### 📊 Cuántas Features Seleccionar?

**Regla general**:
```
Número de features ≈ sqrt(Número de registros) / 10
```

**Ejemplos**:
- 100 registros → ~1 feature
- 1,000 registros → ~3 features
- 10,000 registros → ~10 features
- 100,000 registros → ~30 features

⚠️ **No es una regla estricta**, solo una guía inicial.

**Mejor enfoque**: Usar **validación cruzada** para encontrar el número óptimo.

---

## 2️⃣ Métodos de Filtro (Filter Methods)

### 📖 Concepto

**Métodos de filtro** evalúan cada feature **individualmente** usando métricas estadísticas, **independientes del modelo**.

**Ventajas**:
- ✅ **Rápidos**: No entrenan modelos
- ✅ **Escalables**: Funcionan con miles de features
- ✅ **Independientes del modelo**: Funcionan con cualquier algoritmo

**Desventajas**:
- ❌ **No consideran interacciones**: Evalúan features aisladamente
- ❌ **Pueden descartar features útiles**: En combinación con otras

---

### 📊 Método 1: Varianza Baja (Low Variance)

**Idea**: Eliminar features con **poca variabilidad** (casi constantes).

**¿Por qué?** Si una feature es casi constante, no aporta información.

**Ejemplo**:
```python
feature_A = [1, 1, 1, 1, 1, 1, 1, 1]     # Varianza = 0 → Eliminar
feature_B = [1, 2, 1, 2, 1, 2, 1, 2]     # Varianza > 0 → Mantener
```

**Implementación**:
```python
from sklearn.feature_selection import VarianceThreshold

# Eliminar features con varianza < 0.01
selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X)

print(f"Features originales: {X.shape[1]}")
print(f"Features seleccionadas: {X_selected.shape[1]}")
```

⚠️ **Importante**: Escalar datos antes (StandardScaler) para comparar varianzas.

---

### 📊 Método 2: Correlación con el Target

**Idea**: Seleccionar features **más correlacionadas** con el target.

**Métricas**:
- **Pearson correlation**: Para features **continuas** y relación **lineal**
- **Spearman correlation**: Para relaciones **monotónicas** (no lineales)

**Implementación**:
```python
import pandas as pd

# Calcular correlaciones
correlations = df.corr()['target'].abs().sort_values(ascending=False)

# Seleccionar top 10 features
top_features = correlations[1:11].index.tolist()  # Excluir el target mismo

print("Top 10 features por correlación:")
print(correlations[1:11])
```

**Visualización**:
```python
import seaborn as sns
import matplotlib.pyplot as plt

# Heatmap de correlaciones
plt.figure(figsize=(12, 8))
sns.heatmap(df[top_features + ['target']].corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlación de Top Features con Target')
plt.show()
```

---

### 📊 Método 3: Chi-Cuadrado (χ²)

**Uso**: **Clasificación** con features **categóricas** o **binarias**.

**Idea**: Mide **dependencia** entre feature y target.

**Hipótesis**:
- H₀: Feature y target son **independientes**
- H₁: Feature y target **NO son independientes**

**Implementación**:
```python
from sklearn.feature_selection import SelectKBest, chi2

# Seleccionar top 10 features por chi-cuadrado
selector = SelectKBest(score_func=chi2, k=10)
X_selected = selector.fit_transform(X, y)

# Ver scores
scores = pd.DataFrame({
    'feature': X.columns,
    'chi2_score': selector.scores_
}).sort_values('chi2_score', ascending=False)

print(scores.head(10))
```

⚠️ **Importante**: Features deben ser **no negativas** para chi2.

---

### 📊 Método 4: Mutual Information (MI)

**Idea**: Mide **cuánta información** aporta una feature sobre el target.

**Ventaja**: Captura relaciones **no lineales**.

**Uso**:
- `mutual_info_classif`: Para **clasificación**
- `mutual_info_regression`: Para **regresión**

**Implementación (Regresión)**:
```python
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# Seleccionar top 15 features por MI
selector = SelectKBest(score_func=mutual_info_regression, k=15)
X_selected = selector.fit_transform(X, y)

# Ver scores
scores = pd.DataFrame({
    'feature': X.columns,
    'mi_score': selector.scores_
}).sort_values('mi_score', ascending=False)

print(scores.head(15))
```

**Visualización**:
```python
plt.figure(figsize=(10, 6))
plt.barh(scores['feature'][:15], scores['mi_score'][:15])
plt.xlabel('Mutual Information Score')
plt.title('Top 15 Features por Mutual Information')
plt.gca().invert_yaxis()
plt.show()
```

---

### 📊 Método 5: ANOVA F-test

**Uso**: **Clasificación** con features **continuas**.

**Idea**: Mide si las medias de la feature son **diferentes** entre clases.

**Implementación**:
```python
from sklearn.feature_selection import SelectKBest, f_classif

# Seleccionar top 10 features por F-score
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X, y)

# Ver scores
scores = pd.DataFrame({
    'feature': X.columns,
    'f_score': selector.scores_,
    'p_value': selector.pvalues_
}).sort_values('f_score', ascending=False)

print(scores.head(10))
```

---

### ⚖️ Comparación de Métodos de Filtro

| Método | Tipo Target | Tipo Features | Captura No Linealidad | Velocidad |
|---------|-------------|---------------|------------------------|------------|
| **Varianza** | N/A | Numéricas | N/A | ⭐⭐⭐⭐⭐ |
| **Correlación** | Continuo | Numéricas | ❌ No | ⭐⭐⭐⭐⭐ |
| **Chi²** | Categórico | Categóricas | ✅ Sí | ⭐⭐⭐⭐ |
| **Mutual Info** | Ambos | Ambas | ✅ Sí | ⭐⭐⭐ |
| **ANOVA F** | Categórico | Numéricas | ❌ No | ⭐⭐⭐⭐ |

---

### 💻 Ejemplo Completo: Dataset de Panadería

```python
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_regression

# Cargar datos
df = pd.read_csv('ventas.csv')

# Preparar features y target
X = df[['dia_semana', 'mes', 'sucursal_id', 'h3_index_encoded', 'segmento_encoded']]
y = df['total']

# Seleccionar top 3 features por MI
selector = SelectKBest(score_func=mutual_info_regression, k=3)
X_selected = selector.fit_transform(X, y)

# Ver features seleccionadas
selected_features = X.columns[selector.get_support()].tolist()
print(f"Features seleccionadas: {selected_features}")
```

---